In [1]:
import numpy as np
import pandas as pd
# import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# from xgboost import XGBClassifier
# from impyute.imputation.cs import mice
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
from sklearn.preprocessing import PowerTransformer

# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

warnings.filterwarnings('ignore')

### Read real and syn data

In [2]:
tabular_data = pd.read_csv("data/MIMICDATA_MoreDisease_all_test.csv", index_col="Unnamed: 0") # data/MIMICDATA_200_209_mid_tests.csv data/combined_MIMICDATA_200_209.csv
# syn_file_path = "output/model/2024_06_01_11_07_26_3000_epochs_onto_dp_cgans_model_519_unseen_with_previous_diseases_all_tests_full_penalty.pkl"
# syn_file_path = "output/model/2024_05_06_13_50_43_3000_epochs_onto_dp_cgans_model_519_unseen_with_previous_diseases_all_tests_full_penalty.pkl"
# syn_file_path = "output/model/2024_05_02_19_56_51_3000_epochs_onto_dp_cgans_model_519_unseen_with_previous_diseases_impact_tests_full_penalty.pkl"
thresh = 16

# syn_file_path = "output/model/2024_05_04_00_09_42_3000_epochs_onto_dp_cgans_model_519_unseen_with_previous_diseases_impact_tests_full_penalty.pkl"
syn_file_path = "output/model/2024_05_04_12_04_27_3000_epochs_onto_dp_cgans_model_519_unseen_with_previous_diseases_impact_tests_full_penalty.pkl"
# syn_file_path = "output/model/2024_05_04_12_01_09_3000_epochs_onto_dp_cgans_model_519_unseen_with_previous_7_diseases_impact_tests_full_penalty.pkl"


tabular_data['icd_code']=tabular_data['icd_code'].apply(lambda x: np.floor(x / 10) if x > 9999 else np.floor(x))
tabular_data=tabular_data.drop_duplicates()
tabular_data=tabular_data.drop("subject_id",axis=1)
tabular_data['gender'].replace(0, 'Male',inplace=True)
tabular_data['gender'].replace(1, 'Female',inplace=True)

icd_to_ordo = {
               2040.0: "http://www.orpha.net/ORDO/Orphanet_513", # Acute Lymphoblastic Leukemia (ALL)
               2050.0: "http://www.orpha.net/ORDO/Orphanet_519", # AML	
               2007.0: "http://www.orpha.net/ORDO/Orphanet_544", # Diffuse large B-cell lymphoma
               2041.0: "http://www.orpha.net/ORDO/Orphanet_67038", # B-cell lymphoma
               2387.0: "http://www.orpha.net/ORDO/Orphanet_52688", # Myelodysplastic syndrome (MDS)

               ### 2848.0: "http://www.orpha.net/ORDO/Orphanet_88",  # Other specified aplastic anemias
               ### 2898.0: "http://www.orpha.net/ORDO/Orphanet_824", # Primary myelofibrosis
               ### 2051.0: "http://www.orpha.net/ORDO/Orphanet_521", # Chronic myelogenous leukemia (CML)
               ### 2841.0: "http://www.orpha.net/ORDO/Orphanet_2585", # Pancytopenia
    
               # 2840.0: "http://www.orpha.net/ORDO/Orphanet_68383", # Constitutional Aplastic anemia
               # 2842.0: "http://www.orpha.net/ORDO/Orphanet_314399", # Myelophthisic anemia (MPA)
               
               2005.0: "http://www.orpha.net/ORDO/Orphanet_46135",#	Primary central nervous system lymphoma
               2028.0: "http://www.orpha.net/ORDO/Orphanet_207046", # Malignant lymphoma with peripheral neuropathy		
              }

ordo_list = list(icd_to_ordo.keys())

real_data_ordo = tabular_data[tabular_data["icd_code"].isin(ordo_list)]



# selected_col = ['icd_code', 'gender', 'anchor_age', #'diastolic', 'systolic',
#                 "White Blood Cells", "Red Blood Cells","MCH","MCV","RDW","MCHC",
#                 "Platelet Count","Hematocrit","Hemoglobin","Neutrophils",
#                 "Basophils","Eosinophils","Monocytes",
#                 "PT","INR(PT)","PTT",
#                 "Bicarbonate","Glucose","Magnesium","Calcium, Total","Phosphate",
#                 "Creatinine","Urea Nitrogen","Potassium","Chloride","Sodium",
#                 "Bilirubin, Total","Asparate Aminotransferase (AST)",
#                 "Alanine Aminotransferase (ALT)","Alkaline Phosphatase",
#                 "Lactate Dehydrogenase (LD)","Albumin"]

selected_col = ['icd_code', 'gender', "Neutrophils", "White Blood Cells", #"PTT",
                 "Monocytes", "Platelet Count","Hematocrit", "anchor_age",
                "Red Blood Cells", "PT", "INR(PT)", "MCHC", "Eosinophils", 
                "Hemoglobin", "Basophils", "RDW"]

# selected_col = ['icd_code', 'gender', 'anchor_age', 'diastolic', 'systolic', 'Hematocrit', 'Glucose',
#                  'White Blood Cells', 'Hemoglobin', 'MCHC', 'MCH','MCV','RDW','Lactate Dehydrogenase (LD)',
#                 'Monocytes', 'Bilirubin, Total', 'Eosinophils', 'INR(PT)', 'Basophils',
#                 'Sodium', 'Calcium, Total', 'Bicarbonate', 'PT', 'Red Blood Cells', 'Neutrophils', 
#                 'Platelet Count','Lymphocytes', 
#                 'Reticulocyte Count, Automated', 'Protein, Total', 'Blasts',]
# 
#                 ## 'Nucleated Red Cells' , 'Uric Acid', "Ferritin", "Protein, Total", "Potassium, Whole Blood", "Vitamin B12"


real_data_ordo = real_data_ordo[selected_col]
# real_data_ordo = real_data_ordo.dropna(axis=0, thresh=15)

iri_list = []
for i in real_data_ordo.icd_code:
    iri_list.append(icd_to_ordo[i])
real_data_ordo['IRI']=iri_list

# shift column 'Name' to first position 
first_column = real_data_ordo.pop('IRI') 
real_data_ordo.insert(0, 'IRI', first_column) 

replace_icd = {
               2040.0: "ORDO.Orphanet_513", # Acute Lymphoblastic Leukemia (ALL)
               2050.0: "ORDO.Orphanet_519", # AML
               
               2007.0: "ORDO.Orphanet_544", # Diffuse large B-cell lymphoma               
               2041.0: "ORDO.Orphanet_67038", # B-cell lymphoma
               2387.0: "ORDO.Orphanet_52688", # Myelodysplastic syndrome (MDS)

               ### 2848.0: "ORDO.Orphanet_88",  # Other specified aplastic anemias
               ### 2898.0: "ORDO.Orphanet_824", # Primary myelofibrosis		
               ### 2051.0: "ORDO.Orphanet_521", # Chronic myelogenous leukemia (CML)
               ### 2841.0: "ORDO.Orphanet_2585", # Pancytopenia
               
               2840.0: "ORD..Orphanet_68383", # Constitutional Aplastic anemia
               2842.0: "ORDO.Orphanet_314399", # Myelophthisic anemia
              
               2005.0: "ORDO.Orphanet_46135",#	Primary central nervous system lymphoma
               2028.0: "ORDO.Orphanet_207046", # Malignant lymphoma with peripheral neuropathy	

               
              }

# replace_icd = {
#                2050.0: "AML", #"Acute myeloid leukemia",		
#                2040.0: "ALL", #"Acute Lymphoblastic Leukemia",
#                2840.0: "CAA", #"Constitutional Aplastic anemia",
#                2051.0: "CML", #"Chronic myelogenous leukemia (CML)",
#                2842.0: "MPA", #"Myelophthisic anemia",
#                2848.0: "Other_AA", #"Other specified aplastic anemias",
#                2041.0: "BCLL", #"B-cell chronic lymphocytic leukemia",
#                2051.0: "CML", #"Chronic myelogenous leukemia",
#                2387.0: "MDS", #"Myelodysplastic syndrome",
#                2898.0: "PMF", #"Primary myelofibrosis",
#                2007.0: "DLBCL", # "Diffuse large B-cell lymphoma",
#               }

for i in replace_icd.keys():
    real_data_ordo['icd_code'].replace(i, replace_icd[i],inplace=True)


##### For statistical analysis #####
# #### remove duplicates patients with multiple diseases #####

real_data_ordo['temp_column'] = real_data_ordo['icd_code'].apply(lambda x: 1 if x == 'ORDO.Orphanet_519' else 0)
real_data_ordo = real_data_ordo.sort_values(by='temp_column', ascending=False)
real_data_ordo = real_data_ordo.drop_duplicates(subset=selected_col[1:], keep="first")
real_data_ordo = real_data_ordo.drop("temp_column", axis=1)

##### For machine learning prediction #####
#### remove records from AML patients who have multiple diseases #####
# real_data_ordo_519 = real_data_ordo[real_data_ordo['icd_code']=='ORDO.Orphanet_519']
# real_data_ordo_ml=real_data_ordo_519

# for idx, row in real_data_ordo[selected_col[1:]].iterrows():
#     if not (row == real_data_ordo_519[selected_col[1:]]).all(axis=1).any():
#         real_data_ordo_ml=real_data_ordo_ml.append(real_data_ordo.loc[idx])
# real_data_ordo=real_data_ordo_ml.drop_duplicates(keep='first')
        
real_data_target = real_data_ordo[real_data_ordo.icd_code == "ORDO.Orphanet_519"]
real_data_rest = real_data_ordo[real_data_ordo.icd_code != "ORDO.Orphanet_519"]
print("Other Diseases:", len(real_data_rest))
print("Target Disease:", len(real_data_target))
print("All Disease:", len(real_data_ordo))

Other Diseases: 2056
Target Disease: 268
All Disease: 2324


In [3]:
# real_data_ordo[(real_data_ordo.icd_code=="ORDO.Orphanet_519") | (real_data_ordo.icd_code=="ORDO.Orphanet_52688")]#.drop(['IRI','icd_code'],axis=1).drop_duplicates()

In [4]:
### Remove dulicates patients (with more than 1 disease, we only keep 1 disease from these patients)

# real_data_ordo = real_data_ordo.loc[real_data_ordo.drop(['IRI','icd_code'],axis=1).drop_duplicates().index]

In [5]:
# duplicates = real_data_ordo.drop(['IRI','icd_code'],axis=1).duplicated(keep=False)  # 'keep=False' marks all duplicates as True
# print(len(real_data_ordo[duplicates][['icd_code']]))
# print(real_data_ordo[duplicates][['icd_code']].value_counts())

In [6]:
# plt.figure(figsize=(7,5))
# sns.color_palette("Set2")
# ax = real_data_ordo['icd_code'].value_counts(sort=True).plot.bar(rot=90, width=0.7, )
# ax.bar_label(ax.containers[0])
# plt.rcParams.update({'font.size': 10})
# ### real_data_ordo['icd_code'].value_counts(sort=True)

In [7]:
# grouped = real_data_ordo.groupby(selected_col[1:])
# duplicate_indices = []
# # Iterate over each group
# for name, group in grouped:
#     if len(group) > 1:  # If there's more than one in the group, it's a duplicate
#         dups = real_data_ordo.loc[group.index.tolist()]["icd_code"].tolist()
#         dups.sort()
#         duplicate_indices.append(dups)
# duplicate_indices.sort()
# # duplicate_indices
# multi_diseases = pd.Series(duplicate_indices).value_counts().keys().tolist()
# multi_diseases_count = pd.Series(duplicate_indices).value_counts().values.tolist()
# # multi_diseases

In [8]:
# multi_diseases[0:8]

In [9]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots()
# list_num = multi_diseases_count[0:8]
# # list_num.append(np.sum(multi_diseases_count[8:]))

# fruits = ['207046 & 544', '519 & 52688', '207046 & 46135', '207046 & 52688','207046 & 67038', '207046 & 46135 & 544', 
#           '513 & 519', '521 & 52688']
# counts = list_num

# ax.bar(fruits, counts, color = "coral")

# ax.bar_label(ax.containers[0])
# plt.xticks(rotation=90)
# plt.show()

In [10]:
# from tableone import TableOne
# import pandas as pd
# real_data_ordo['target'] = np.where(real_data_ordo['icd_code'] == 'ORDO.Orphanet_519', 1, 0)

# categorical = ['gender']
# nonnormal = ['diastolic', 'systolic', 'anchor_age']#columns[3:]
# groupby = ['target']
# mytable = TableOne(real_data_ordo[real_data_ordo.columns[2:]], groupby=groupby, pval=True)
# # print(mytable.tabulate(tablefmt = "github"))
# mytable

### EV Patients

In [11]:
# E_patient_data = pd.read_csv("data/MIMICDATA_E8859_patient_test.csv", index_col="Unnamed: 0")
# E_patient_data = E_patient_data[selected_col[1:]]
# E_patient_data_cleaned = E_patient_data.dropna(axis=0, thresh=9)

## remove weird data points 
# E_patient_data_cleaned = E_patient_data_cleaned[E_patient_data_cleaned['BMI']<=60]
# E_patient_data_cleaned = E_patient_data_cleaned[E_patient_data_cleaned['White Blood Cells']<=100]
# E_patient_data_cleaned = E_patient_data_cleaned[E_patient_data_cleaned['Lactate Dehydrogenase (LD)']<=1000]
# E_patient_data_cleaned = E_patient_data_cleaned[E_patient_data_cleaned['Absolute Lymphocyte Count']<=60]

# E_patient_data_cleaned.describe()

In [12]:
def sample_syn_data(generator_path, n_rows, picked_unseen_rds):
    
    synthesizer = CTGANSynthesizer.load(generator_path)
    if len(picked_unseen_rds) > 0:
        sampled_data = synthesizer.sample(n_rows, unseen_rds=picked_unseen_rds)
    elif len(picked_unseen_rds) == 0:
        sampled_data = synthesizer.sample(n_rows)
        
    print(f'Sampling {n_rows} unseen rows') 
    return sampled_data

In [13]:
n_rows = len(real_data_target) * 10  #real_data_target
picked_unseen_rds = ["http://www.orpha.net/ORDO/Orphanet_52688",]

# unseen_large_pen_data = sample_syn_data("output/model/2024_02_22_14_42_03_1500_epochs_onto_dp_cgans_model_penalty_large.pkl",
                                        # n_rows,
                                        # picked_unseen_rds)

unseen_super_large_pen_data = sample_syn_data(syn_file_path, #("output/model/2024_03_08_14_08_25_1500_epochs_onto_dp_cgans_model_519_unseen_with_9_diseases_all_tests_100penalty.pkl",
                                        n_rows,
                                        picked_unseen_rds)

### ("output/model/2024_03_10_11_27_12_3000_epochs_onto_dp_cgans_model_519_unseen_with_8_diseases_all_tests_full_penalty_withBlasts_noicdcode.pkl"
### output/model/2024_03_09_11_09_28_3000_epochs_onto_dp_cgans_model_519_unseen_with_9_diseases_all_tests_full_penalty_withBlasts.pkl
### output/model/2024_03_07_14_38_45_1500_epochs_onto_dp_cgans_model_519_unseen_with_9_diseases_all_tests_10penalty.pkl
### output/model/2024_03_07_12_06_44_1500_epochs_onto_dp_cgans_model_519_unseen_with_9_diseases_half_tests.pkl
### all variables all diseases ### output/model/2024_03_06_17_26_08_2000_epochs_onto_dp_cgans_model_519_unseen_with_more_relevant_diseases.pkl
### half variable all diseases ### output/model/2024_03_07_11_19_48_2000_epochs_onto_dp_cgans_model_519_unseen_with_more_relevant_diseases_half_tests.pkl

# other_data = sample_syn_data("output/model/2024_02_22_16_35_07_1500_epochs_onto_dp_cgans_model_penalty_super_large.pkl",
#                                         n_rows,
#                                         ["http://www.orpha.net/ORDO/Orphanet_908"])


# # output/model/2024_02_22_14_42_03_1500_epochs_onto_dp_cgans_model_penalty_large.pkl
# output/model/2024_02_22_16_35_07_1500_epochs_onto_dp_cgans_model_penalty_super_large.pkl
# output/model/2024_02_29_16_20_57_1500_epochs_onto_dp_cgans_model_521_unseen.pkl
# output/model/2024_02_29_17_20_49_1500_epochs_onto_dp_cgans_model_521_unseen.pkl
# output/model/2024_03_01_00_48_23_2000_epochs_onto_dp_cgans_model_544_unseen.pkl
# output/model/2024_03_01_09_49_18_2000_epochs_onto_dp_cgans_model_207046_unseen.pkl

# unseen_no_pen_data = sample_syn_data("output/model/2024_03_03_15_25_37_1500_epochs_onto_dp_cgans_model_207046_unseen_no_penalty.pkl",
#                                         n_rows,
#                                         picked_unseen_rds)

ctgan_data = sample_syn_data("trained_generators/generator_ctgan_519_220240508_epo_2000.pkl",
                                        400,
                                        [])

# gpt_data = pd.read_csv("data/gpt_aml_data.csv")
# gpt_data.columns = real_data_target.columns
# gpt_data


SamplingError: This synthesizer was created on a machine with GPU but the current machine is CPU-only. This feature is currently unsupported. We recommend sampling on the same GPU-enabled machine.

In [ ]:
unseen_super_large_pen_data=unseen_super_large_pen_data.dropna(axis=0, thresh=thresh) #34
ctgan_data = ctgan_data.dropna(axis=0, thresh=thresh) #34
print("number of synthetic target data", len(unseen_super_large_pen_data))
print("number of synthetic ctgan target data", len(ctgan_data))

In [ ]:
real_data_target = real_data_target[selected_col[0:]].drop_duplicates()
real_data_rest = real_data_rest[selected_col[0:]].drop_duplicates()

unseen_super_large_pen_data = unseen_super_large_pen_data[selected_col[0:]]
# other_data = other_data[selected_col[1:]]
# unseen_no_pen_data = unseen_no_pen_data[selected_col[1:]]
ctgan_data = ctgan_data[selected_col[0:]]


### Diagnotic: Perform basic checks to ensure the synthetic data is valid
### Data quality: Compare the real and synthetic data's statistical similarity.

In [ ]:
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality
from sdv.metadata import SingleTableMetadata

def evaluationReport(target_data, syn_data):
    
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(data=target_data)
    
    diagnostic_report = run_diagnostic(real_data=target_data, synthetic_data=syn_data, metadata=metadata)
    quality_report = evaluate_quality(real_data=target_data, synthetic_data=syn_data, metadata=metadata)
    

    # diagnostic_report.generate(target_data, syn_data, metadata.to_dict())
    # diagnostic_report.get_properties()
    
    
    # quality_report.generate(target_data, syn_data, metadata.to_dict())
    # quality_report.get_properties()
    # quality_report.get_details(property_name='Column Shapes')

    return diagnostic_report, quality_report

In [ ]:
# diagnostic_report, quality_report = evaluationReport(real_data_target[selected_col[2:]], unseen_super_large_pen_data[selected_col[2:]])#ctgan_data)#real_data_rest)#unseen_super_large_pen_data)
# diagnostic_report.get_details(property_name='Coverage') #'Coverage', 'Boundary', 'Synthesis'.
# quality_report.get_details(property_name='Column Shapes')

In [ ]:
# corr_df = quality_report.get_details(property_name='Column Pair Trends')
# corr_df.to_csv("corr_compare.csv")

In [ ]:
# diagnostic_report, quality_report = evaluationReport(real_data_target[selected_col[1:]], ctgan_data[selected_col[1:]])
# diagnostic_report.get_details(property_name='Coverage') #'Coverage', 'Boundary', 'Synthesis'.
# quality_report.get_details(property_name='Column Shapes')

In [ ]:
# dsf

In [ ]:
# from sdmetrics.visualization import get_column_plot

# column_name = real_data_target.columns[13]#"Creatinine, Urine"
# fig = get_column_plot(
#     real_data=real_data_target,
#     synthetic_data=real_data_rest,
#     column_name=column_name,
#     plot_type='distplot',
# )

# # fig.update_layout(
# #     autosize=False,
# #     width=800,
# #     height=400,
# # )
# # fig.update_layout(xaxis_range=[5,16])
# fig.show()

# fig = get_column_plot(
#     real_data=real_data_target,
#     synthetic_data=unseen_super_large_pen_data,
#     column_name=column_name,
#     plot_type='distplot',
# )

# # fig.update_layout(
# #     autosize=False,
# #     width=800,
# #     height=400,
# # )

In [ ]:
# select_col = real_data_target.columns[[2, 5, 7, 9]] # 2, 5, 7, 31 'RDW'
# data_labels = ["Real AML data", "Real Other diseases", "Syn AML (OntoCGAN)"]
# datasets = [real_data_target, real_data_rest, unseen_super_large_pen_data]
# plot_df = pd.DataFrame()
# for i in range(0,3):
#     temp_df = datasets[i][select_col]
#     temp_df['label'] = [data_labels[i]] * len(temp_df)
#     plot_df = plot_df.append(temp_df)
# plot_df = plot_df.reset_index(drop=True)
# plot_df


# select_col = "Platelet Count" #"Hematocrit"
 
# fig, ax = plt.subplots()
# sns.kdeplot(real_data_target[select_col], color="slategrey", shade=True, linewidth=1.8, alpha=.3)#, bw_adjust=2.5)
# sns.kdeplot(real_data_rest[select_col], color="steelblue", shade=True, linewidth=1.2, alpha=.2)
# sns.kdeplot(ctgan_data[select_col], color="indianred", shade=True, linewidth=1.2, alpha=.2)
# sns.kdeplot(unseen_super_large_pen_data[select_col], color="green", shade=True, linewidth=1.2, alpha=.2)

# plt.legend(["Real AML", "Real Others", "CTGAN", "Syn AML (OntoCGAN)"])
# plt.grid(color = 'grey', linestyle = '--', linewidth = 0.5)
# # ax.set_xlim(-200,1200)
# fig.savefig("output/dist_platelet.png") 

In [ ]:
# def corr_plot(dataframe, syndata=None, vmin=-0.5, vmax=0.5):
#     plt.grid()
#     if type(syndata) !=pd.core.frame.DataFrame:
#         corr = dataframe.corr()
#         cmap = sns.cubehelix_palette(start=.8, rot=-.8, as_cmap=True) # sns.diverging_palette(220, 20, as_cmap=True)
#         mask = np.triu(np.ones_like(corr, dtype=bool))
#         corr_plot = sns.heatmap(corr, annot=False, cmap=cmap, vmin=vmin, vmax=vmax, mask=mask)
#     else:
#         corr = abs(dataframe.corr()-syndata.corr())
#         cmap = sns.color_palette("Blues", as_cmap=True) 
#         mask = np.triu(np.ones_like(corr, dtype=bool))
#         corr_plot = sns.heatmap(corr, annot=True, cmap=cmap, vmin=0, vmax=0.7, mask=mask)
    

#     return corr_plot

# # corr_col = real_data_target.columns[3:16]
# corr_col = ['White Blood Cells', 'Red Blood Cells', 'MCH', 'MCV', 'RDW', 'MCHC',
#         'Hematocrit', 'Hemoglobin',
#        'Basophils', 'Eosinophils', 'Monocytes']
# corr_plot(real_data_target[corr_col], None, -0.5, 0.5)#,unseen_super_large_pen_data[real_data_target.columns[2:16]])

# corr_plot(unseen_super_large_pen_data[corr_col], None, -0.25, 0.25)

### Prediction evaluation
Train on healthy + patients data

The Scenario is we don't have 519 patients. So we train the classifiers on data with healthy people + synthetic 519 patients
Test it on real dataset

In [ ]:
### people with other diseases
negative_class_data = real_data_rest

### Healthy people 
# negative_class_data = E_patient_data_cleaned#.sample(len(real_data_rest))

negative_class_data['target'] = [0] * len(negative_class_data)

positive_class_data = real_data_target
positive_class_data['target'] = [1] * len(positive_class_data)

full_training_data = negative_class_data.append(positive_class_data)

In [ ]:
# Impupte missing values in real data

categorial_var = ["icd_code", "gender"]# ["gender"] ["icd_code", "gender"]
imputed_real = mice(full_training_data.drop(categorial_var,axis=1).values)
imputed_real = pd.DataFrame(imputed_real, columns=full_training_data.columns.drop(categorial_var), index=full_training_data.index)
imputed_real['gender'] = full_training_data['gender']
imputed_real['icd_code'] = full_training_data['icd_code']

### Impute missing values in syn data

syn_class_data = unseen_super_large_pen_data #unseen_super_large_pen_data #unseen_no_pen_data #ctgan_data #unseen_super_large_pen_data
syn_class_data['target'] = [2] * len(syn_class_data)
try: 
    imputed_syn = mice(syn_class_data.drop(categorial_var,axis=1).values)
    imputed_syn = pd.DataFrame(imputed_syn, columns=syn_class_data.columns.drop(categorial_var), index=syn_class_data.index)
    imputed_syn['gender'] = syn_class_data['gender']
    imputed_syn['icd_code'] = syn_class_data['icd_code']
except:
    imputed_syn = syn_class_data
    

In [ ]:
imputed_data = imputed_real.append(imputed_syn)#(syn_class_data)
imputed_data = imputed_data.reset_index(drop=True)

imputed_data['gender'].replace('Male', 0,inplace=True)
imputed_data['gender'].replace('Female', 1,inplace=True)

# GPT Generated data ###
## syn_class_data = gpt_data
## syn_class_data['target'] = [2] * len(gpt_data)

In [ ]:
# syn_class_data = unseen_super_large_pen_data #unseen_no_pen_data #ctgan_data #unseen_super_large_pen_data
# syn_class_data['target'] = [2] * len(syn_class_data)

# full_training_data = full_training_data.append(syn_class_data)

In [ ]:
# categorial_var = ["gender",'icd_code']
# imputed_data = mice(full_training_data.drop(categorial_var,axis=1).values)
# imputed_data = pd.DataFrame(imputed_data, columns=full_training_data.columns.drop(categorial_var), index=full_training_data.index)
# imputed_data['gender'] = full_training_data['gender']
# imputed_data['icd_code'] = full_training_data['icd_code']

# imputed_data = imputed_data.reset_index(drop=True)

# imputed_data['gender'].replace('Male', 0,inplace=True)
# imputed_data['gender'].replace('Female', 1,inplace=True)

In [ ]:
# features = full_training_data.columns[:-1]  # Assuming you want to plot the first 20 features
# linewidth = 0.1
# alpha = 0.1

# plt.figure(figsize=(20, 60))
# for i, feature in enumerate(features, 1):
#     plt.subplot(15, 2, i)  # Adjust grid size depending on the number of features
#     sns.histplot(real_data_target[feature], kde=True, stat="density", label="Target", alpha=alpha, linewidth = linewidth)
#     sns.histplot(real_data_rest[feature], kde=True, stat="density", label="rest", alpha=alpha, linewidth = linewidth)
#     # sns.histplot(gpt_data[feature], kde=True, stat="density", label="GPT4", alpha=alpha, linewidth = linewidth)
#     sns.histplot(unseen_super_large_pen_data[feature], kde=True, stat="density", label="onto_cgan",alpha=alpha, linewidth = linewidth)
#     # sns.histplot(ctgan_data[feature], kde=True, stat="density", label="ctgan_data",alpha=alpha, linewidth = linewidth)
#     plt.legend()
#     # plt.title(feature)
# plt.tight_layout()
# plt.show()

In [ ]:
# from impyute.imputation.cs import mice

# categorial_var = ["gender"]
# # imputed_data = full_training_data.drop(categorial_var,axis=1)
# imputed_data = mice(full_training_data.drop(categorial_var,axis=1).values)
# imputed_data = pd.DataFrame(imputed_data, columns=full_training_data.columns.drop(categorial_var), index=full_training_data.index)
# imputed_data['gender'] = full_training_data['gender']


### Single classification

In [ ]:
def confusion_map(y_test, preds):
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(4,3))
    plt.clf()
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.tab20c)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ['Negative','Positive'])
    plt.yticks(tick_marks, ['Negative','Positive'])
    s = [['TN','FP'], ['FN', 'TP']]
  
    for i in range(2):
        for j in range(2):
            plt.text(j,i, str(s[i][j])+" = "+str(cm[i][j]))
    plt.show()

In [ ]:
### Add syn to real 
def add_syn_to(X, y, imputed_data):
    X_syn = X.loc[imputed_data['target'][imputed_data['target'] == 0].keys()]
    y_syn = y.loc[imputed_data['target'][imputed_data['target'] == 0].keys()]

    X_syn = X_syn.append(imputed_data[imputed_data['target']==2].drop(["target","icd_code"],axis=1), ignore_index=True)
    # y_syn = y_syn.append(imputed_data[imputed_data['target']==2]['target'], ignore_index=True).replace(2,1)
    y_syn = pd.Series(['ORDO.Orphanet_519'] * len(X_syn))

    return X_syn, y_syn

### Get all positive cases for testing

### Train (add syn) test on real

In [ ]:
def prepare_data_for_ontocgan(X_train, y_train, X_test, y_test, imputed_data):
    # X_test_real = X_test.append(X_train.loc[y_train[y_train == 1].keys()], ignore_index=True)
    # y_test_real = y_test.append(y_train.loc[y_train[y_train == 1].keys()], ignore_index=True)
    
    X_test_real = X_test.append(X_train.loc[y_train[y_train == 'ORDO.Orphanet_519'].keys()], ignore_index=True)
    y_test_real = y_test.append(y_train.loc[y_train[y_train == 'ORDO.Orphanet_519'].keys()], ignore_index=True)
    y_test_all_real_notarget = y_test_real.replace('ORDO.Orphanet_519', 'ORDO.Orphanet_519_other')
    
    ### Train (add syn) test on real###
    ### Train on real+syn unseen data, test on real
    ## Keep negative cases in real data + unseen syn data
    # X_train_syn = X_train.loc[y_train[y_train == 0].keys()]
    # y_train_syn = y_train.loc[y_train[y_train == 0].keys()]
    
    X_train_syn = X_train.loc[y_train[y_train != 'ORDO.Orphanet_519'].keys()]
    y_train_syn = y_train.loc[y_train[y_train != 'ORDO.Orphanet_519'].keys()]
    # #Add unseen syn data to the training
    # X_train_syn = X_train_syn.append(transformed_data[imputed_data['target']==2], ignore_index=True)
    X_train_syn = X_train_syn.append(imputed_data[imputed_data['target']==2].drop(["target","icd_code"],axis=1), ignore_index=True)
    # y_train_syn = y_train_syn.append(imputed_data[imputed_data['target']==2]['target'], ignore_index=True).replace(2,1)
    y_train_syn = y_train_syn.append(pd.Series(['ORDO.Orphanet_519'] * len(imputed_data[imputed_data['target']==2])), ignore_index=True)
    
    # X_test_syn = X_test.append(X_train_syn.loc[y_train_syn[y_train_syn == 1].keys()], ignore_index=True)
    # y_test_syn = y_test.append(y_train_syn.loc[y_train_syn[y_train_syn == 1].keys()], ignore_index=True)
    
    X_test_syn = X_test.append(X_train_syn.loc[y_train_syn[y_train_syn == 'ORDO.Orphanet_519'].keys()], ignore_index=True)
    y_test_syn = y_test.append(y_train_syn.loc[y_train_syn[y_train_syn == 'ORDO.Orphanet_519'].keys()], ignore_index=True)

    return X_train_syn, y_train_syn, X_test_syn, y_test_syn

In [ ]:
# X_train_real_notarget = X_train.loc[y_train[y_train != 'ORDO.Orphanet_519'].keys()]
# y_train_real_notarget = y_train.loc[y_train[y_train != 'ORDO.Orphanet_519'].keys()]

# # otherDisease = tabular_data[(tabular_data.icd_code==2841) | (tabular_data.icd_code==2030) | (tabular_data.icd_code==2898) | 
# # (tabular_data.icd_code==2019) | (tabular_data.icd_code==2020) | (tabular_data.icd_code==2021)]
# # addedOn_otherDisease = otherDisease[X_train.columns].dropna(axis=0, thresh=26).sample(200)

# otherDisease = X_train.loc[y_train[y_train == 'ORDO.Orphanet_519'].keys()].sample(1)
# addedOn_otherDisease= otherDisease

# X_train_real_notarget = X_train_real_notarget.append(addedOn_otherDisease, ignore_index=True)
# y_train_real_notarget = y_train_real_notarget.append(pd.Series(['ORDO.Orphanet_519_other'] * len(addedOn_otherDisease)), ignore_index=True)
# X_train_real_notarget['gender'].replace('Male', 0,inplace=True)
# X_train_real_notarget['gender'].replace('Female', 1,inplace=True)

# y_test_real_notarget = y_test.replace('ORDO.Orphanet_519', 'ORDO.Orphanet_519_other')

In [ ]:
def prepare_data_for_ctgan(X_train, y_train, ctgan_data):

    try: 
        imputed_syn = mice(ctgan_data.drop(['gender','icd_code'],axis=1).values)
        imputed_syn = pd.DataFrame(imputed_syn, columns=ctgan_data.columns.drop(['gender','icd_code']), index=ctgan_data.index)
        imputed_syn['gender'] = ctgan_data['gender']
        imputed_syn['icd_code'] = ctgan_data['icd_code']
    except:
        imputed_syn = ctgan_data

    
    X_train_ctgan = X_train.loc[y_train[y_train != 'ORDO.Orphanet_519'].keys()]
    y_train_ctgan = y_train.loc[y_train[y_train != 'ORDO.Orphanet_519'].keys()]
    
    X_train_ctgan = X_train_ctgan.append(imputed_syn.drop(["icd_code"],axis=1), ignore_index=True)
    y_train_ctgan = y_train_ctgan.append(pd.Series(['ORDO.Orphanet_519'] * len(imputed_syn)), ignore_index=True)
    X_train_ctgan['gender'].replace('Male', 0,inplace=True)
    X_train_ctgan['gender'].replace('Female', 1,inplace=True)
    
    y_test_ctgan = y_test.replace('ORDO.Orphanet_519', 'ORDO.Orphanet_519_ctgan')

    return X_train_ctgan, y_train_ctgan, y_test_ctgan
   

In [ ]:
from sklearn.preprocessing import label_binarize

def classification(X_train, y_train, X_test, y_test):

    # bst = XGBClassifier(objective='multi:softprob', n_estimators=500, learning_rate=1e-4)
    # bst = sklearn.neighbors.KNeighborsClassifier(n_neighbors=100)
    bst = sklearn.ensemble.RandomForestClassifier(n_estimators=500, class_weight='balanced', criterion="gini")
    # bst = sklearn.naive_bayes.GaussianNB(var_smoothing=1e-8)
    # bst = sklearn.linear_model.LogisticRegression(max_iter=300, class_weight='balanced',
    #                                               solver="lbfgs",penalty='l2',multi_class='multinomial', intercept_scaling=50) #‘newton-cg’, ‘sag’, ‘saga’ and ‘lbfgs’
    

    
    bst.fit(X_train, y_train)
    # model_filename = f'model_LR.pkl'
    # joblib.dump(bst, model_filename)
    
    # make predictions
    preds = bst.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    cm_display = ConfusionMatrixDisplay(cm, display_labels=bst.classes_).plot(cmap=sns.color_palette("ch:s=.25,rot=-.25", as_cmap=True),)
    plt.xticks(rotation=90)
    print()
    
    print('ROC_AUC_score', roc_auc_score(y_test, bst.predict_proba(X_test), multi_class='ovo'))#[:, 1]
    print('PR_AUC_score', average_precision_score(y_test, bst.predict_proba(X_test)))#[:, 1]


    # print('ROC_AUC_score_519', roc_auc_score(y_test, bst.predict_proba(X_test)[:, 2], multi_class='ovo'))[:, 1]
    # print('PR_AUC_score_519', average_precision_score(y_test, bst.predict_proba(X_test)[:, 2]))[:, 1]
    
    # Binarize the labels for one-vs-rest calculation
    y_true_binarized = label_binarize(y_test, classes=bst.classes_)
    target_class = list(bst.classes_).index("ORDO.Orphanet_519")
    
    # Calculate ROC AUC for a specific class, say class 0
    roc_auc = roc_auc_score(y_true_binarized[:, target_class], bst.predict_proba(X_test)[:, target_class])
    prauc = average_precision_score(y_true_binarized[:, target_class], bst.predict_proba(X_test)[:, target_class])
    print("ROC_AUC_score_519", roc_auc)
    print("PR_AUC_score_519", prauc)
    
    print(classification_report(y_test, preds, target_names=bst.classes_))

    return bst

def roc_curve_plots(clf_list, X_test, y_test):
    fig, ax = plt.subplots(figsize=(4, 4))
    
    for item in range(0, len(clf_list)):
        y_true_binarized = label_binarize(y_test, classes=clf_list[item].classes_)
        target_class = list(clf_list[item].classes_).index("ORDO.Orphanet_519")
    
        # # Calculate ROC AUC for a specific class, say class 0
        # roc_auc = roc_auc_score(y_true_binarized[:, target_class], bst.predict_proba(X_test)[:, target_class])
    
        y_score = clf_list[item].predict_proba(X_test)[:, target_class]
        fpr, tpr, _ = roc_curve(y_test, y_score, pos_label=clf_list[item].classes_[target_class])
        # roc_display = RocCurveDisplay(fpr=fpr, tpr=tpr).plot()

        plt.plot(
            fpr,
            tpr,
            label=f"Classifier {item}: (AUC = {auc(fpr, tpr):.3f})",
            linewidth=1,
        )
    _ = ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="ROC AUC",)
    ax.legend()

def pr_curve_plots(clf_list, X_test, y_test):
    
    fig, ax = plt.subplots(figsize=(4, 4))

    for item in range(0, len(clf_list)):
        if item == len(clf_list)-1 :
            plot_chance_level = True
        else:
            plot_chance_level = False
            
        # y_pred = clf_list[item].predict(X_test)#[:, 1]
        y_score = clf_list[item].predict_proba(X_test)#[:, 1]
        PrecisionRecallDisplay.from_predictions(
            y_test, y_score, name=f"Classifier {item}", ax=ax, plot_chance_level=plot_chance_level,
            pos_label=1
        )
    plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, roc_auc_score, classification_report, accuracy_score, roc_curve, confusion_matrix, average_precision_score, precision_recall_curve
from matplotlib import *
from sklearn.metrics import auc, RocCurveDisplay, roc_curve
from sklearn.metrics import PrecisionRecallDisplay, precision_recall_curve
from pylab import *
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import PrecisionRecallDisplay

# transformed_data = PowerTransformer(method='yeo-johnson').fit_transform(imputed_data.drop(['target', 'icd_code'],axis=1))
# transformed_data = pd.DataFrame(transformed_data, columns=imputed_data.drop(['target', 'icd_code'],axis=1).columns)

### Train on real data
X = imputed_data[imputed_data['target']!=2].drop(['target','icd_code'],axis=1) # imputed_data ['target','icd_code']
# X = transformed_data[imputed_data['target']!=2]
y = imputed_data[imputed_data['target']!=2]['icd_code']
# y = imputed_data[imputed_data['target']!=2]['target']  # Your target # imputed_data #['icd_code']


# Using KFold for splitting
kfold = StratifiedKFold(n_splits=5, shuffle=False, random_state=None)

# Perform 5-fold cross-validation
for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y)):
    if fold == 4:
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        print(f"Fold {fold+1}:")
    
        X_train_syn, y_train_syn, X_test_syn, y_test_syn = prepare_data_for_ontocgan(X_train, y_train, X_test, y_test, imputed_data)
        # X_train_ctgan, y_train_ctgan, y_test_ctgan = prepare_data_for_ctgan(X_train, y_train, ctgan_data)
        
        # clf_only_real = classification(X_train, y_train, X_test, y_test)
        clf_train_syn_test_real = classification(X_train_syn, y_train_syn, X_test, y_test)
    
        # clf_train_ctgan = classification(X_train_ctgan, y_train_ctgan, X_test, y_test)
    
        print()



# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=None)
# X_train_syn, y_train_syn, X_test_syn, y_test_syn = prepare_data_for_ontocgan(X_train, y_train, X_test, y_test, imputed_data)
# bst = sklearn.ensemble.RandomForestClassifier(n_estimators=500, class_weight='balanced', criterion="gini")
# clf_only_real = classification(bst, X_train, y_train, X_test, y_test)
# bst = sklearn.ensemble.RandomForestClassifier(n_estimators=500, class_weight='balanced', criterion="gini")
# clf_train_syn_test_real = classification(bst, X_train_syn, y_train_syn, X_test, y_test)

In [ ]:
##### individual test ####
# clf_only_real = classification(X_train, y_train, X_test, y_test)
# clf_train_syn_test_real = classification(X_train_syn, y_train_syn, X_test, y_test)
# clf_train_no_target = classification(X_train_real_notarget, y_train_real_notarget, X_test, y_test_real_notarget)
# clf_train_ctgan = classification(X_train_ctgan, y_train_ctgan, X_test, y_test)
# clf_train_real_syn_test_real = classification(X_train_syn.append(X_train), y_train_syn.append(y_train), X_test, y_test)

In [ ]:
sns.heatmap(clf_only_real.predict_proba(X_test.loc[y_test[y_test=="ORDO.Orphanet_519"].keys()]), cmap=sns.color_palette("ch:s=.25,rot=-.25", as_cmap=True), annot=False, vmax=0.7)

In [ ]:
sns.heatmap(clf_train_syn_test_real.predict_proba(X_test.loc[y_test[y_test=="ORDO.Orphanet_519"].keys()]), cmap=sns.color_palette("ch:s=.25,rot=-.25", as_cmap=True), annot=False, vmax=0.7)

In [ ]:
# sns.heatmap(clf_train_ctgan.predict_proba(X_test.loc[y_test[y_test=="ORDO.Orphanet_519"].keys()]), cmap=sns.color_palette("ch:s=.25,rot=-.25", as_cmap=True), annot=False, vmax=0.7)

In [ ]:
----- STOP -----

In [ ]:
----- STOP -----

In [ ]:
----- STOP -----

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Assuming you have your dataset X_train and y_train

# Initialize Random Forest Classifier
rf = RandomForestClassifier()

# Train the model
rf.fit(X_train, y_train)

# Retrieve feature importances
feature_importances = rf.feature_importances_

# Create a DataFrame to store feature names and their importance scores
feature_importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': feature_importances})

# Sort the DataFrame by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Print the feature importance ranking
feature_importance_df

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Assuming you have your dataset X_train and y_train

# Initialize Random Forest Classifier
rf = RandomForestClassifier()

# Train the model
rf.fit(X_train_syn, y_train_syn)

# Retrieve feature importances
feature_importances = rf.feature_importances_

# Create a DataFrame to store feature names and their importance scores
feature_importance_df = pd.DataFrame({'Feature': X_train_syn.columns, 'Importance': feature_importances})

# Sort the DataFrame by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Print the feature importance ranking
print(feature_importance_df)

In [ ]:
# print('ROC_AUC', roc_auc_score(y_test_real, clf_only_real.predict(X_test_real)))
# print('ROC_AUC', roc_auc_score(y_test_real, clf_train_syn_test_real.predict(X_test_real)))

In [ ]:
# roc_curve_plots([clf_only_real,clf_train_syn_test_real], X_test, y_test)
# pr_curve_plots([clf_only_real,clf_train_syn_test_real], X_test, y_test)

In [ ]:
# import shap
# # clf_train_syn_test_real, clf_only_real
# explainer = shap.Explainer(
#     clf_train_syn_test_real, X_train_syn, feature_names=X_train_syn.columns
# )
# shap_values = explainer(X_test)
# shap.plots.beeswarm(shap_values,max_display=10)#[:,:,1])

In [ ]:
# import shap
# # clf_train_syn_test_real, clf_only_real
# explainer = shap.Explainer(
#     clf_only_real, X_train, feature_names=X_train.columns
# )
# shap_values = explainer(X_test)
# shap.plots.beeswarm(shap_values,max_display=10)#[:,:,1])

In [ ]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import precision_score, recall_score, ConfusionMatrixDisplay, f1_score, roc_auc_score, classification_report, accuracy_score, roc_curve, confusion_matrix, average_precision_score, precision_recall_curve


# def stratified_cv(X, y, model, cv=5, test_mode=None, X_real=None, y_real=None):
#     skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=None)
#     scores = {"Accuracy":[],
#         "Precision":[],
#         "Recall":[],
#         "F1_score":[],
#         "Roc_auc":[],
#         "PR_auc":[]}

#     for iteration in range(0, 1):
    
#         if test_mode == "test sub-real":
            
#             X_train_list = []
#             y_train_list = []
    
#             X_test_syn_list = []
#             y_test_syn_list = []
#             for train_index, test_index in skf.split(X, y):
#                 X_train, X_test = X.loc[train_index], X.loc[test_index]
#                 y_train, y_test = y.loc[train_index], y.loc[test_index]
    
#                 X_train_list.append(X_train)
#                 y_train_list.append(y_train)
#                 X_test_syn_list.append(X_test)
#                 y_test_syn_list.append(y_test)
                
#             x_test_real_list = []
#             y_test_real_list = []
#             for train_index, test_index in skf.split(X_real, y_real):
#                 X_train, X_test = X_real.loc[train_index], X_real.loc[test_index]
#                 y_train, y_test = y_real.loc[train_index], y_real.loc[test_index]
            
#                 x_test_real_list.append(X_test.loc[y_test[y_test == 1].keys()])
#                 y_test_real_list.append(y_test.loc[y_test[y_test == 1].keys()])
    
#             for i in range(0, cv):
#                 X_test = X_test_syn_list[i].loc[y_test_syn_list[i][y_test_syn_list[i] == 0].keys()]
#                 y_test = y_test_syn_list[i].loc[y_test_syn_list[i][y_test_syn_list[i] == 0].keys()]
#                 X_test = X_test.append(x_test_real_list[i], ignore_index=True)
#                 y_test = y_test.append(y_test_real_list[i], ignore_index=True)
    
    
#                 # print("training", len(X_train_list[i]), len(X_test), y_test.value_counts())
                    
#                 model.fit(X_train_list[i], y_train_list[i])
#                 predictions = model.predict(X_test)
#                 predictions_scores = model.predict_proba(X_test)[:, 1]
#                 average_method = 'binary'
                
#                 scores['Accuracy'].append(accuracy_score(y_test, predictions))
#                 scores['Precision'].append(precision_score(y_test, predictions, average=average_method))
#                 scores['Recall'].append(recall_score(y_test, predictions, average=average_method))
#                 scores['F1_score'].append(f1_score(y_test, predictions, average=average_method))
#                 scores['Roc_auc'].append(roc_auc_score(y_test, predictions_scores, average='macro'))
#                 scores['PR_auc'].append(average_precision_score(y_test, predictions_scores, average='macro'))

    
    
#         else:
#             for train_index, test_index in skf.split(X, y):
#                 X_train, X_test = X.loc[train_index], X.loc[test_index]
#                 y_train, y_test = y.loc[train_index], y.loc[test_index]
        
#                 if test_mode == "test all":
#                     X_test = X_test.append(X_train.loc[y_train[y_train == 1].keys()], ignore_index=True)
#                     y_test = y_test.append(y_train.loc[y_train[y_train == 1].keys()], ignore_index=True)
#                 elif test_mode == "test real":
#                     X_test = X_test.loc[y_test[y_test == 0].keys()]
#                     y_test = y_test.loc[y_test[y_test == 0].keys()]
                    
#                     X_test = X_test.append(X_real.loc[y_real[y_real == 1].keys()], ignore_index=True)
#                     y_test = y_test.append(y_real.loc[y_real[y_real == 1].keys()], ignore_index=True)            
            
    
#                 # print("training", len(X_train), len(X_test), y_test.value_counts())
                    
#                 model.fit(X_train, y_train)
#                 predictions = model.predict(X_test)
#                 predictions_scores = model.predict_proba(X_test)[:, 1]
#                 average_method = 'binary'
                
#                 scores['Accuracy'].append(accuracy_score(y_test, predictions))
#                 scores['Precision'].append(precision_score(y_test, predictions, average=average_method))
#                 scores['Recall'].append(recall_score(y_test, predictions, average=average_method))
#                 scores['F1_score'].append(f1_score(y_test, predictions, average=average_method))
#                 scores['Roc_auc'].append(roc_auc_score(y_test, predictions_scores, average='macro'))
#                 scores['PR_auc'].append(average_precision_score(y_test, predictions_scores, average='macro'))

#     return scores